# Week 4 · FlashAttention — Tiling & Online Softmax

**Repository:** `bitwise-llm-forge` · **Theory doc:** [`docs/theory/week4_attention.md`](../docs/theory/week4_attention.md)

---

### Learning objectives

1. State why standard attention is **bandwidth-bound**, not compute-bound, on modern GPUs.
2. Derive the **online softmax** recurrence and prove it is numerically equivalent to the offline formulation.
3. Read and run a pure-PyTorch reproduction of FlashAttention's **tiled** algorithm.
4. Verify that the tiled output matches naive attention to floating-point tolerance.
5. Compare three attention variants — naive, tiled, linear — across sequence lengths and visualize the scaling.


In [ ]:
# Make ``src/`` importable when this notebook is launched from anywhere.
from pathlib import Path
import sys

_here = Path.cwd()
for candidate in (_here, *_here.parents):
    if (candidate / "src").is_dir():
        sys.path.insert(0, str(candidate))
        break

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from src.utils.seeding import set_seed
set_seed(42)

print("environment ready · torch", __import__("torch").__version__)


## 1 · Mathematical derivation

### 1.1 Why naive attention is wasteful

The standard scaled dot-product

$$
\operatorname{Attn}(Q,K,V) = \operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right) V
$$

materializes the $N\times N$ score matrix $S=QK^\top$ in **HBM** (high-bandwidth memory). For $N=32{,}768$, just $S$ is $\sim 2$ GB in FP16 — far larger than on-chip SRAM. The kernel spends most of its wall-time **shuttling $S$ between HBM and SRAM**.

### 1.2 GPU memory hierarchy (A100 numbers)

| Memory | Size | Bandwidth |
|---|---|---|
| Registers | a few KB / thread | ∞ |
| SRAM (shared)| ~ 100 KB / SM | ~ 19 TB/s |
| HBM | 40–80 GB | ~ 2 TB/s |

Every byte you can keep in SRAM costs ~ $10\times$ less time than the same byte in HBM.

### 1.3 Online softmax recurrence

Streaming softmax over blocks $z^{(1)}, z^{(2)}, \dots$ maintains a running max $m$ and denominator $\ell$:

$$
m^{(j)} = \max(m^{(j-1)}, \tilde m^{(j)}), \qquad
\ell^{(j)} = e^{m^{(j-1)} - m^{(j)}}\,\ell^{(j-1)} + e^{\tilde m^{(j)} - m^{(j)}}\,\tilde\ell^{(j)}.
$$

This is exact, numerically stable, and works one block at a time.

### 1.4 FlashAttention tiling

Outer loop over $Q$-blocks of size $B_r$, inner loop over $K, V$-blocks of size $B_c$. The block sizes are chosen so that one $Q$, one $K$, one $V$, and a partial output fit together in SRAM:

$$
2B_r d + 2B_c d \;\le\; M_{\text{SRAM}}.
$$

HBM reads drop from $\Theta(N^2)$ (naive) to $\Theta(N^2 d^2 / M)$ (tiled) — typically a $10$–$100\times$ reduction.


## 2 · Reference implementations

In [ ]:
import torch
import torch.nn.functional as F
import time, json
from pathlib import Path
import matplotlib.pyplot as plt

from src.attention import NaiveAttention, TiledAttention, LinearAttention

torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

d_model, n_heads, seq_len = 64, 4, 32
x = torch.randn(2, seq_len, d_model, device=device)

naive = NaiveAttention(d_model, n_heads, causal=True).to(device).eval()
tiled = TiledAttention(d_model, n_heads, block_size_q=8, block_size_kv=8, causal=True).to(device).eval()
linear = LinearAttention(d_model, n_heads).to(device).eval()

# Copy weights so they compute the *same function* (only K/V flow differs).
tiled.qkv_proj.load_state_dict(naive.qkv_proj.state_dict())
tiled.o_proj.load_state_dict(naive.o_proj.state_dict())

with torch.no_grad():
    y_n = naive(x); y_t = tiled(x); y_l = linear(x)

print("output shapes:", tuple(y_n.shape), tuple(y_t.shape), tuple(y_l.shape))


## 3 · Numerical equivalence: tiled vs naive

In [ ]:
# Tiled attention must reproduce naive attention to fp32 tolerance.
diff = (y_n - y_t).abs()
print(f"max abs diff (tiled vs naive) : {float(diff.max().item()):.2e}")
print(f"rms diff                       : {float(diff.pow(2).mean().sqrt().item()):.2e}")
assert torch.allclose(y_n, y_t, atol=1e-4), "tiled attention diverged from naive!"
print("\n✓ tiled attention matches naive attention within 1e-4")


## 4 · Visualising the attention pattern

In [ ]:
# Pull out the raw attention scores from the naive implementation for one head
# and look at the resulting causal triangular structure.
with torch.no_grad():
    qkv = naive.qkv_proj(x).reshape(2, seq_len, 3, n_heads, d_model // n_heads)
    q, k, _ = qkv.unbind(dim=2)
    q = q.transpose(1, 2); k = k.transpose(1, 2)
    scores = (q @ k.transpose(-2, -1)) / (d_model // n_heads) ** 0.5
    mask = torch.ones(seq_len, seq_len, dtype=torch.bool).triu(diagonal=1)
    scores = scores.masked_fill(mask.to(device), float("-inf"))
    attn = torch.softmax(scores, dim=-1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].imshow(scores[0, 0].cpu().numpy(), cmap="viridis")
axes[0].set_title("Raw scores (head 0) before softmax")
axes[1].imshow(attn[0, 0].cpu().numpy(), cmap="viridis")
axes[1].set_title("Attention weights (head 0) after softmax")
for ax in axes:
    ax.set_xlabel("key position"); ax.set_ylabel("query position")
plt.tight_layout(); plt.show()


## 5 · Benchmark — naive vs tiled vs linear vs SDPA

For each sequence length we measure forward-pass median latency and peak memory.
On CUDA we also record `torch.cuda.max_memory_allocated`.


In [ ]:
def median_time(fn, warmup: int = 2, iters: int = 5) -> float:
    for _ in range(warmup): fn()
    if device.type == "cuda": torch.cuda.synchronize()
    samples = []
    for _ in range(iters):
        t0 = time.perf_counter()
        fn()
        if device.type == "cuda": torch.cuda.synchronize()
        samples.append((time.perf_counter() - t0) * 1000.0)
    samples.sort(); return samples[len(samples) // 2]


def measure(layer, x):
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    ms = median_time(lambda: layer(x))
    if device.type == "cuda":
        peak = torch.cuda.max_memory_allocated() / (1024 ** 2)
    else:
        peak = float("nan")
    return ms, peak


# Smaller default seq-lengths for CPU to keep the notebook quick.
seq_lengths = [128, 256, 512, 1024, 2048] if device.type == "cuda" else [64, 128, 256, 512]

# Build modules once
mods = {
    "naive":  NaiveAttention(d_model, n_heads).to(device).eval(),
    "tiled":  TiledAttention(d_model, n_heads, block_size_q=64, block_size_kv=64).to(device).eval(),
    "linear": LinearAttention(d_model, n_heads).to(device).eval(),
}

rows = []
with torch.no_grad():
    for N in seq_lengths:
        x = torch.randn(1, N, d_model, device=device)
        row = {"N": N}
        for name, mod in mods.items():
            try:
                ms, mb = measure(mod, x)
                row[f"{name}_ms"] = ms
                row[f"{name}_mb"] = mb
            except (RuntimeError, torch.cuda.OutOfMemoryError):
                row[f"{name}_ms"] = float("nan")
                row[f"{name}_mb"] = float("nan")
                if device.type == "cuda": torch.cuda.empty_cache()
        rows.append(row)
        print(f"N={N:>5} | "
              f"naive {row['naive_ms']:6.2f} ms | "
              f"tiled {row['tiled_ms']:6.2f} ms | "
              f"linear {row['linear_ms']:6.2f} ms")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ns = [r["N"] for r in rows]

for name, marker in (("naive", "o"), ("tiled", "s"), ("linear", "^")):
    axes[0].plot(ns, [r[f"{name}_ms"] for r in rows], "-"+marker, label=name)
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("sequence length N"); axes[0].set_ylabel("latency (ms)")
axes[0].set_title("Forward latency vs N"); axes[0].grid(True, which="both", alpha=0.3); axes[0].legend()

if device.type == "cuda":
    for name, marker in (("naive", "o"), ("tiled", "s"), ("linear", "^")):
        axes[1].plot(ns, [r[f"{name}_mb"] for r in rows], "-"+marker, label=name)
    axes[1].set_xscale("log"); axes[1].set_yscale("log")
    axes[1].set_xlabel("sequence length N"); axes[1].set_ylabel("peak GPU MB")
    axes[1].set_title("Peak GPU memory vs N")
    axes[1].grid(True, which="both", alpha=0.3); axes[1].legend()
else:
    axes[1].axis("off"); axes[1].text(0.5, 0.5, "(memory profile requires CUDA)",
                                       ha="center", va="center", transform=axes[1].transAxes)

plt.tight_layout(); plt.show()

# Persist
Path("../benchmarks/results").mkdir(parents=True, exist_ok=True)
Path("../benchmarks/results/attention_scaling.json").write_text(json.dumps(rows, indent=2))
print("results saved to ../benchmarks/results/attention_scaling.json")


## 6 · Discussion

- **Pure-PyTorch tiling is not faster** than naive attention — the FlashAttention win comes from *custom CUDA kernels* that keep the inner loop inside SRAM. The reproduction here is **algorithmic**: it expresses the right block structure and is numerically identical, but each block still round-trips through PyTorch tensor allocation.
- **Linear attention is the asymptote.** For very large $N$, the kernel-trick reordering wins decisively in both time and memory — at the cost of an approximate softmax. Many production long-context models combine: tiled exact attention for short contexts, linear attention for ultra-long.
- **Why `F.scaled_dot_product_attention` is your friend.** Since PyTorch 2.0, `F.scaled_dot_product_attention` dispatches to FlashAttention-2 (or efficient memory-attention) automatically on supported hardware. In production, prefer it over rolling your own.
- **FlashAttention-2's contribution.** v1 already had the right algorithm; v2 (Dao, 2023) restructures the *backward pass* loop to avoid non-coalesced HBM writes, giving a further $\sim 2\times$ speedup on training.

### References
- Dao, T. et al. "FlashAttention: Fast and Memory-Efficient Exact Attention." NeurIPS 2022.
- Dao, T. "FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning." arXiv:2307.08691, 2023.
- Milakov, M. & Gimelshein, N. "Online normalizer calculation for softmax." arXiv:1805.02867, 2018.
